# 画像・動画・音声・3DデータをW&Bへ記録する

第2章 Modelsのサンプルです。上から順に実行してください。

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nejumi/wandb_fun_book_public/blob/main/notebooks/multimedia_logging.ipynb)

保存先は接続設定の `ENTITY`（team名）と `PROJECT` で指定できます。APIキーは `wandb.login()` の画面から入力してください。

In [ ]:
%pip install -q "wandb>=0.31,<0.32" datasets soundfile pillow requests numpy pandas matplotlib plotly scikit-learn moviepy opencv-python-headless rdkit

In [ ]:
# W&Bへの接続は次のセルで設定します。

In [ ]:
import os
import wandb

ENTITY = os.environ.get("WANDB_ENTITY", "")  # 保存先のteam名。空欄なら既定値を使用
PROJECT = os.environ.get("WANDB_PROJECT", "book-v4-multimedia")
if ENTITY:
    os.environ["WANDB_ENTITY"] = ENTITY
wandb.login()

from pathlib import Path
import requests

# サンプルで使う小さなメディアファイルだけ取得します。
sample_dir = Path("sample_data")
sample_dir.mkdir(exist_ok=True)
for filename in ["openai-gym.mp4", "piano.wav", "wolf.obj", "some_html.html"]:
    target = sample_dir / filename
    if not target.exists():
        url = f"https://raw.githubusercontent.com/wandb/examples/main/examples/data/{filename}"
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        target.write_bytes(response.content)


In [ ]:
# ログインは上のセルで完了しています。

# 1 画像のロギング
まずはマルチメディアのロギングの基本として、画像のロギングを行いましょう。画像のロギングには主に以下の3つの方法があります:
  1. PIL.Imageオブジェクトを使用
  2. numpy arrayを使用
  3. 画像ファイルのパスを使用



これらのいずれかをwandb.Image()に渡してwandb.Imageオブジェクトを作成し、wandb.log()でロギングします。

## 1-1 PIL.Imageオブジェクトを渡すやり方

In [ ]:
import wandb
from PIL import Image
import requests
from io import BytesIO

# 新しいrunを開始
run = wandb.init(project=PROJECT, name="simple_image")

# 1. 画像のロギング
# オンライン画像をダウンロード
response = requests.get("https://raw.githubusercontent.com/wandb/examples/main/examples/data/car_day.jpg", timeout=60)
response.raise_for_status()
img = Image.open(BytesIO(response.content))
wandb.log({"sample_image": wandb.Image(img, caption="サンプル画像")})

run.finish()

## 1-2 numpy arrayを渡すやり方

In [ ]:
import wandb
import numpy as np
import requests
from io import BytesIO
from PIL import Image

# 新しいrunを開始
run = wandb.init(project=PROJECT, name="numpy_image")

# 1. 画像のロギング
# オンライン画像をダウンロード
response = requests.get("https://raw.githubusercontent.com/wandb/examples/main/examples/data/car_day.jpg", timeout=60)
response.raise_for_status()
img = Image.open(BytesIO(response.content))

# PIL.Imageをnumpyアレイに変換
img_array = np.array(img)
print(img_array.shape)

# numpyアレイとしてwandb.Imageに渡す
wandb.log({"sample_image": wandb.Image(img_array, caption="サンプル画像")})

run.finish()

## 1-3 ファイルパスを渡すやり方

In [ ]:
import wandb
import requests
import os

# wandbの実行を開始
run = wandb.init(project=PROJECT, name="image_path")

# 画像をダウンロード
url = "https://raw.githubusercontent.com/wandb/examples/main/examples/data/car_day.jpg"
response = requests.get(url, timeout=60)
response.raise_for_status()

# ファイルを直接保存
temp_file_path = "temp_image.jpg"
with open(temp_file_path, "wb") as f:
    f.write(response.content)

# ファイルパスを使用してwandb.Imageオブジェクトを作成し、ログに記録
wandb.log({"sample_image": wandb.Image(temp_file_path, caption="サンプル画像")})

# wandbの実行を終了
wandb.finish()

# 2 イメージオーバーレイ
画像は単独でもロギングできますが、segmentation maskおよびbounding boxをセットでロギングしてUI上で様々な分析に活用することができます。

## 2-1 まずはBounding Boxをオーバーレイしてみましょう

### 2-1-1 書き方その１{"minX", "maxX", "minY", "maxY"}

In [ ]:
import wandb
from datasets import load_dataset
from PIL import Image
import io

# W&Bの初期化
wandb.init(project=PROJECT, name="coco-example-option-1")

# COCOデータセットの読み込み
dataset = load_dataset("detection-datasets/coco", split="train", streaming=True)
class_names = dataset.features["objects"]["category"].feature.names

# 最初の画像を取得
sample = next(iter(dataset))

# 画像データはすでにPIL Imageオブジェクトとして提供されているようです
image = sample['image']

# バウンディングボックスの作成
boxes = {
    "ground_truth": {
        "box_data": [
            {
                "position": {
                    "minX": box[0] / image.width,
                    "minY": box[1] / image.height,
                    "maxX": (box[0] + box[2]) / image.width,
                    "maxY": (box[1] + box[3]) / image.height
                },
                "class_id": category,
                "box_caption": class_names[category]
            } for box, category in zip(sample['objects']['bbox'], sample['objects']['category'])
        ],
        "class_labels": {i: label for i, label in enumerate(class_names)}
    }
}

# W&Bに画像とバウンディングボックスをログ
wandb_image = wandb.Image(image, boxes=boxes)
wandb.log({"coco_image": wandb_image})

wandb.finish()

### 2-1-2 書き方その２{"middle", "width", "height"}

In [ ]:
import wandb
from datasets import load_dataset
from PIL import Image

# W&Bの初期化
wandb.init(project=PROJECT, name="coco-example-option-2")

# COCOデータセットの読み込み
dataset = load_dataset("detection-datasets/coco", split="train", streaming=True)
class_names = dataset.features["objects"]["category"].feature.names

# 最初の画像を取得
sample = next(iter(dataset))

# 画像データはすでにPIL Imageオブジェクトとして提供されています
image = sample['image']

# バウンディングボックスの作成
boxes = {
    "ground_truth": {
        "box_data": [
            {
                "position": {
                    "middle": [
                        (box[0] + box[2] / 2) / image.width,
                        (box[1] + box[3] / 2) / image.height
                    ],
                    "width": box[2] / image.width,
                    "height": box[3] / image.height
                },
                "class_id": category,
                "box_caption": class_names[category],
                "domain": "relative"
            } for box, category in zip(sample['objects']['bbox'], sample['objects']['category'])
        ],
        "class_labels": {i: label for i, label in enumerate(class_names)}
    }
}

# W&Bに画像とバウンディングボックスをログ
wandb_image = wandb.Image(image, boxes=boxes)
wandb.log({"coco_image": wandb_image})

wandb.finish()

### 2-1-3 Predictionも一緒にログする
BBoxは複数ログできますので、正解データに加えて予測結果もログしましょう。ここでは簡単のためにground_truthにランダムノイズを加えて予測結果に見立てます。また、scoresにconfidenceを与えて、UI上からフィルタできるようにしておきます。

In [ ]:
import wandb
from datasets import load_dataset
from PIL import Image
import numpy as np

# W&Bの初期化
wandb.init(project=PROJECT, name="coco-example-with-predictions")

# COCOデータセットの読み込み
dataset = load_dataset("detection-datasets/coco", split="train", streaming=True)
class_names = dataset.features["objects"]["category"].feature.names

# 最初の画像を取得
sample = next(iter(dataset))

# 画像データ
image = sample['image']

# バウンディングボックスの作成（ground truthとprediction）
boxes = {
    "ground_truth": {
        "box_data": [],
        "class_labels": {i: label for i, label in enumerate(class_names)}
    },
    "predictions": {
        "box_data": [],
        "class_labels": {i: label for i, label in enumerate(class_names)}
    }
}

for box, category in zip(sample['objects']['bbox'], sample['objects']['category']):
    # Ground truth
    gt_box = {
        "position": {
            "minX": box[0] / image.width,
            "minY": box[1] / image.height,
            "maxX": (box[0] + box[2]) / image.width,
            "maxY": (box[1] + box[3]) / image.height
        },
        "class_id": category,
        "box_caption": class_names[category]
    }
    boxes["ground_truth"]["box_data"].append(gt_box)

    # Prediction (ground truth + noise)
    noise = np.random.normal(0, 0.05, 4)  # ノイズの生成
    pred_box = {
        "position": {
            "minX": max(0, min(1, gt_box["position"]["minX"] + noise[0])),
            "minY": max(0, min(1, gt_box["position"]["minY"] + noise[1])),
            "maxX": max(0, min(1, gt_box["position"]["maxX"] + noise[2])),
            "maxY": max(0, min(1, gt_box["position"]["maxY"] + noise[3]))
        },
        "class_id": category,
        "box_caption": class_names[category],
        "scores": {
            "confidence": np.random.uniform(0.5, 1.0)  # ランダムな信頼度スコア
        }
    }
    boxes["predictions"]["box_data"].append(pred_box)

# W&Bに画像とバウンディングボックスをログ
wandb_image = wandb.Image(image, boxes=boxes)
wandb.log({"coco_image": wandb_image})

wandb.finish()

## 2-2 セグメンテーションマスク

In [ ]:
import wandb
import numpy as np
from PIL import Image
import os
from pathlib import Path
import random

# データのダウンロードと展開
if not os.path.exists('BDD100K_seg_demo.zip'):
    response = requests.get('https://storage.googleapis.com/wandb_datasets/BDD100K_seg_demo.zip', timeout=120)
    response.raise_for_status()
    Path('BDD100K_seg_demo.zip').write_bytes(response.content)

if not os.path.exists('segment_demo'):
    import zipfile
    with zipfile.ZipFile('BDD100K_seg_demo.zip') as archive:
        archive.extractall('.')
else:
    print("segment_demo ディレクトリは既に存在します。展開をスキップします。")

# W&Bの初期化
wandb.init(project=PROJECT, name="bdd-example")

# データパスの設定
path_data = Path('segment_demo')
path_img = path_data / 'images' / 'val'
path_lbl = path_data / 'labels' / 'val'

# セグメンテーションクラス
segmentation_classes = [
    'road', 'sidewalk', 'building', 'wall', 'fence', 'pole', 'traffic light',
    'traffic sign', 'vegetation', 'terrain', 'sky', 'person', 'rider', 'car',
    'truck', 'bus', 'train', 'motorcycle', 'bicycle', 'void'
]

# クラスラベルの辞書を作成
class_labels = {i: label for i, label in enumerate(segmentation_classes)}

# ランダムな画像を選択
img_files = list(path_img.glob('*.jpg'))
sample_img_file = random.choice(img_files)

# 画像とマスクを読み込む
img = Image.open(sample_img_file)
mask = Image.open(path_lbl / (sample_img_file.stem + '_train_id.png'))

# マスクデータを整形
mask_data = np.array(mask)

# W&Bに画像とセグメンテーションマスクをログ
wandb_mask = wandb.Image(
    img,
    masks={
        "ground_truth": {
            "mask_data": mask_data,
            "class_labels": class_labels
        }
    }
)
wandb.log({"segmentation_example": wandb_mask})

wandb.finish()

# 3 動画のロギング

動画のロギングはファイルパスないしnumpy arrayを渡します。

## 3-1 ファイルパス指定

In [ ]:
# Runの初期化
wandb.init(project=PROJECT, name="videos")

# ビデオファイル
path_to_video = "sample_data/openai-gym.mp4"

# ロギング
wandb.log({"video": wandb.Video(path_to_video, fps=4, format="gif")})

wandb.finish()

## 3-2 numpy arrayによる方法

In [ ]:
import wandb
import numpy as np
import cv2

# Runの初期化
wandb.init(project=PROJECT, name="videos-from-array")

# ビデオファイルの読み込み
path_to_video = "sample_data/openai-gym.mp4"
cap = cv2.VideoCapture(path_to_video)

frames = []
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    # OpenCVはBGR形式で読み込むので、RGBに変換
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frames.append(frame)

cap.release()

# NumPy配列に変換
video_array = np.array(frames)

# 配列の形状を表示
print(f"ビデオ配列の形状: {video_array.shape}")

# W&Bの期待する形式に変換 (T, C, H, W)
video_array = np.transpose(video_array, (0, 3, 1, 2))

print(f"変換後のビデオ配列の形状: {video_array.shape}")

# W&Bにログ
wandb.log({"video_from_array": wandb.Video(video_array, fps=30, format="gif")})

wandb.finish()

# 4 音声データのロギング

## 4-1 ファイルパス指定

In [ ]:
# Initialize a new run
wandb.init(project=PROJECT, name="audio_path")

# Generate audio data
path_to_audio = "sample_data/piano.wav"

# Log that audio data
wandb.log({"examples":
           [wandb.Audio(path_to_audio, caption="Piano")]})

wandb.finish()

## 4-2 numpy arrayを渡す

In [ ]:
# Initialize a new run
wandb.init(project=PROJECT, name="audio_array")

# Generate audio data
fs = 44100 # sampling frequency, Hz
length = 3  # length, seconds
xs = np.linspace(0, length, num=fs * length)
waveform = np.sin(fs * 2 * np.pi / 40  * xs ** 2)

# Log audio data
wandb.log({"examples":
           [wandb.Audio(waveform, caption="Boop", sample_rate=fs)]})

wandb.finish()

# 5 3Dオブジェクト/ポイントクラウド

## 5-1 3Dオブジェクト
対応フォーマットは`'obj', 'gltf', 'glb', 'babylon', 'stl', 'pts.json'`です。

In [ ]:
# Initialize a new run
wandb.init(project=PROJECT, name="3d_objects")

# Generate 3D object data
path_to_obj = "sample_data/wolf.obj"

# Log the 3D object
wandb.log({"3d_object": wandb.Object3D(open(path_to_obj))})

wandb.finish()

## 5-2 ポイントクラウド

In [ ]:
# Initialize a new run
wandb.init(project=PROJECT, name="point_clouds")

# Generate a cloud of points
points = np.random.uniform(size=(250, 3))

# Log points and boxes in W&B
wandb.log(
        {
            "point_scene": wandb.Object3D(
                {
                    "type": "lidar/beta",
                    "points": points,
                    "boxes": np.array(
                        [
                            {
                                "corners": [
                                    [0,0,0],
                                    [0,1,0],
                                    [0,0,1],
                                    [1,0,0],
                                    [1,1,0],
                                    [0,1,1],
                                    [1,0,1],
                                    [1,1,1]
                                ],
                                "label": "Box",
                                "color": [123,255,111],
                            },
                            {
                                "corners": [
                                    [0,0,0],
                                    [0,2,0],
                                    [0,0,2],
                                    [2,0,0],
                                    [2,2,0],
                                    [0,2,2],
                                    [2,0,2],
                                    [2,2,2]
                                ],
                                "label": "Box-2",
                                "color": [111,255,0],
                            }
                        ]
                    ),
                    "vectors": np.array([])
                }
            )
        }
    )

wandb.finish()

## 5-3 LiDARデータで両者を同時にログしてみる

In [ ]:
# 任意：nuScenes miniを用意済みの場合だけ実行します。
# データの取得先：https://www.nuscenes.org/nuscenes#download
NUSCENES_ROOT = ""  # v1.0-mini、samplesなどがあるディレクトリ
if NUSCENES_ROOT:
    %pip install -q nuscenes-devkit


In [ ]:
# nuScenes miniの実データを使う追加例です。
if not NUSCENES_ROOT:
    print("nuScenesの追加例はスキップします。実行する場合は上のNUSCENES_ROOTを設定してください。")
else:
    import numpy as np
    from nuscenes.nuscenes import NuScenes
    from nuscenes.utils.data_classes import LidarPointCloud
    from pyquaternion import Quaternion

    nusc = NuScenes(version="v1.0-mini", dataroot=NUSCENES_ROOT, verbose=False)
    sample = nusc.sample[0]
    lidar_data = nusc.get("sample_data", sample["data"]["LIDAR_TOP"])
    ego_pose = nusc.get("ego_pose", lidar_data["ego_pose_token"])
    sensor = nusc.get("calibrated_sensor", lidar_data["calibrated_sensor_token"])
    pc = LidarPointCloud.from_file(nusc.get_sample_data_path(lidar_data["token"]))
    boxes = []
    for token in sample["anns"]:
        annotation = nusc.get("sample_annotation", token)
        box = nusc.get_box(token)
        box.translate(-np.array(ego_pose["translation"]))
        box.rotate(Quaternion(ego_pose["rotation"]).inverse)
        box.translate(-np.array(sensor["translation"]))
        box.rotate(Quaternion(sensor["rotation"]).inverse)
        boxes.append({"corners": box.corners().T.tolist(), "label": annotation["category_name"], "color": [255, 255, 255]})
    with wandb.init(project=PROJECT, name="lidar-demo") as run:
        run.log({"point_scene": wandb.Object3D({"type": "lidar/beta", "points": pc.points[:3].T, "boxes": np.array(boxes)})})


# 6 HTMLのロギング

In [ ]:
# Initialize a new run
wandb.init(project=PROJECT, name="html")

# Generate HTML data
path_to_html = "sample_data/some_html.html"

# Log an HTML file
wandb.log({"custom_file": wandb.Html(open(path_to_html))})

# Log raw HTML strings
wandb.log({"custom_string": wandb.Html('<a href="https://mysite">Link</a>')})

wandb.finish()

# 7 分子構造のロギング

## 7-1 低分子の化学構造（rdkitインテグレーション）
この例ではアセトアミノフェンのSMILESから3D構造の生成と最適化、2D構造の画像生成を行い、それぞれロギングしています。また、wandb.Tableの中に格納してロギングも行っています。wandb.Tableへの格納は構造化データと組み合わせて様々な分析が可能になるため、他のメディアに対しても広く有用です。

In [ ]:
# RDKitは冒頭のセルでインストール済みです。

In [ ]:
import wandb
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
from PIL import Image
import io

# W&Bの初期化
wandb.init(project=PROJECT, name="acetaminophen-demo")

# アセトアミノフェンのSMILES文字列
smiles = "CC(=O)Nc1ccc(O)cc1"

# 3D構造の生成と最適化
mol_3d = Chem.MolFromSmiles(smiles)
mol_3d = Chem.AddHs(mol_3d)
AllChem.EmbedMolecule(mol_3d)
AllChem.MMFFOptimizeMolecule(mol_3d)

# 3D分子をW&Bにログ
wandb.log({"acetaminophen_3d": wandb.Molecule.from_rdkit(mol_3d)})

# 2D構造の生成
mol_2d = Chem.MolFromSmiles(smiles)
AllChem.Compute2DCoords(mol_2d)
img = Draw.MolToImage(mol_2d, size=(300, 300))

# 2D構造をW&Bにログ
wandb.log({"acetaminophen_2d": wandb.Image(img)})

# テーブルの作成
table = wandb.Table(columns=["SMILES", "3D_Structure", "2D_Structure"])
table.add_data(smiles,
               wandb.Molecule.from_rdkit(mol_3d),
               wandb.Image(img))

# テーブルをW&Bにログ
wandb.log({"molecule_table": table})

# runの終了
wandb.finish()

## 7-2 タンパク質の立体構造

In [ ]:
import wandb
import requests
import os

# W&Bの初期化
wandb.init(project=PROJECT, name="hemoglobin-demo")

# ヘモグロビンのPDBファイルをダウンロード
pdb_id = "1GZX"  # ヘモグロビンのPDB ID
url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
response = requests.get(url, timeout=60)
response.raise_for_status()

# PDBファイルを一時的に保存
with open(f"{pdb_id}.pdb", "wb") as f:
    f.write(response.content)

# PDBファイルをW&Bにログ
wandb.log({"hemoglobin_structure": wandb.Molecule(f"{pdb_id}.pdb")})

# テーブルの作成
table = wandb.Table(columns=["PDB_ID", "Protein_Name", "Structure"])
table.add_data(pdb_id, "Hemoglobin", wandb.Molecule(f"{pdb_id}.pdb"))

# テーブルをW&Bにログ
wandb.log({"protein_table": table})

# 一時ファイルを削除
os.remove(f"{pdb_id}.pdb")

# runの終了
wandb.finish()

# その他、プロット類

In [ ]:
import wandb
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.metrics import precision_recall_curve, roc_curve

# W&Bの初期化
wandb.init(project=PROJECT, name="various-plots-example")

# 1. 基本的な線グラフ
x = np.linspace(0, 10, 100)
y = np.sin(x)
data = [[x_val, y_val] for (x_val, y_val) in zip(x, y)]
table = wandb.Table(data=data, columns=["x", "y"])
wandb.log({"line_plot": wandb.plot.line(table, "x", "y", title="Sin(x) Line Plot")})

# 2. 散布図
x = np.random.rand(100)
y = np.random.rand(100)
data = [[x_val, y_val] for (x_val, y_val) in zip(x, y)]
table = wandb.Table(data=data, columns=["x", "y"])
wandb.log({"scatter_plot": wandb.plot.scatter(table, "x", "y", title="Random Scatter Plot")})

# 3. 棒グラフ
labels = ["A", "B", "C", "D"]
values = [23, 45, 56, 78]
data = [[label, value] for (label, value) in zip(labels, values)]
table = wandb.Table(data=data, columns=["label", "value"])
wandb.log({"bar_chart": wandb.plot.bar(table, "label", "value", title="Sample Bar Chart")})

# 4. ヒストグラム
data = np.random.normal(0, 1, 1000)
wandb.log({"histogram": wandb.plot.histogram(wandb.Table(data=[[d] for d in data], columns=["values"]), "values")})

# 5. PR曲線と6. ROC曲線（2クラス分類の例）
y_true = np.random.randint(2, size=100)
y_scores = np.random.random(100)

## PRカーブ
precision, recall, _ = precision_recall_curve(y_true, y_scores)
wandb.log({"pr_curve": wandb.plot.pr_curve(
    y_true,
    np.stack([1 - y_scores, y_scores], axis=1),
    labels=["Negative", "Positive"],
    classes_to_plot=[1]  # Positiveクラスのみプロット
)})

## ROCカーブ
fpr, tpr, _ = roc_curve(y_true, y_scores)
wandb.log({"roc_curve": wandb.plot.roc_curve(
    y_true,
    np.stack([1 - y_scores, y_scores], axis=1),
    labels=["Negative", "Positive"],
    classes_to_plot=[1]  # Positiveクラスのみプロット
)})

# 7. 混同行列
y_true = np.random.randint(3, size=100)
y_pred = np.random.randint(3, size=100)
wandb.log({"confusion_matrix": wandb.plot.confusion_matrix(probs=None,
                                                           y_true=y_true,
                                                           preds=y_pred,
                                                           class_names=["A", "B", "C"])})

# 8. Matplotlibプロット
plt.plot([1, 2, 3, 4])
plt.ylabel("some interesting numbers")
wandb.log({"chart": plt})

wandb.finish()